# Kentucky Weather — Feature Engineering for LSTM Evaluation

This notebook reproduces **exactly** the feature engineering pipeline from the Indiana LSTM training notebook, applied to the Kentucky weather dataset.

**Goal**: Prepare `X` and `y` arrays ready to feed into the pre-trained Indiana LSTM model for cross-region accuracy evaluation.

### Features engineered (identical to training)
| Category | Features |
|---|---|
| Cyclical time | `hour_sin/cos`, `dayofweek_sin/cos`, `month_sin/cos`, `dayofyear_sin/cos` |
| Binary flags | `is_weekend`, `is_business_hour` |
| Lag features | `lag_1h`, `lag_2h`, `lag_3h`, `lag_24h`, `lag_48h`, `lag_168h` |
| Rolling stats | `roll_mean_24h`, `roll_std_24h`, `roll_mean_168h`, `roll_std_168h` |
| Interaction | `temp_rh_interaction` |

> **Note**: The Kentucky dataset contains only weather data (no energy consumption column `aep_mw`). The lag and rolling features that were computed from `aep_mw` during training **cannot** be computed here without Kentucky energy data. Those 10 columns (`lag_*` and `roll_*`) will be created as **zero-filled placeholders** so the feature matrix shape matches the model's expected 29 features. See the dedicated section for details and alternatives.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")


Libraries loaded.


## 2. Configuration — Update Paths Here

In [2]:
# ── UPDATE THESE PATHS ──────────────────────────────────────────────────────
KENTUCKY_WEATHER_PATH = '../data/processed/kentucky_weather_2005_2018.csv'   # Kentucky weather CSV

# Paths to the scalers saved from the Indiana training notebook
FEATURE_SCALER_PATH   = '../../01_campus_substation/models/feature_scaler.pkl'   # saved as joblib from training
TARGET_SCALER_PATH    = '../../01_campus_substation/models/target_scaler.pkl'    # saved as joblib from training

# Output path for the engineered feature matrix
OUTPUT_CSV_PATH       = '../data/processed/kentucky_features_engineered.csv'
# ────────────────────────────────────────────────────────────────────────────

# Constants (must match training exactly)
SEQ_LEN = 168   # 1-week look-back window used during training

# Exact feature column order from training (MUST NOT change)
FEATURE_COLS = [
    'Temp_C', 'Dew_Point_C', 'RH_percent', 'Pressure_kPa',
    'Wind_Dir_deg', 'Wind_Speed_mps', 'Visibility_km', 'Precip_cm',
    'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos',
    'month_sin', 'month_cos', 'dayofyear_sin', 'dayofyear_cos',
    'is_weekend', 'is_business_hour',
    'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'lag_48h', 'lag_168h',
    'roll_mean_24h', 'roll_std_24h',
    'roll_mean_168h', 'roll_std_168h',
    'temp_rh_interaction'
]

print(f"Feature count: {len(FEATURE_COLS)} (must be 29)")
assert len(FEATURE_COLS) == 29, "Feature count mismatch — check FEATURE_COLS!"


Feature count: 29 (must be 29)


## 3. Load & Clean Kentucky Weather Data

In [3]:
df = pd.read_csv(KENTUCKY_WEATHER_PATH, parse_dates=['datetime'])
df['datetime'] = df['datetime'].dt.round('h')   # same rounding as training merge
df = df.sort_values('datetime').reset_index(drop=True)

print(f"Shape         : {df.shape}")
print(f"Date range    : {df['datetime'].min()}  →  {df['datetime'].max()}")
print(f"Missing values: {df.isnull().sum().sum()}")
print()
print(df.head(3))


Shape         : (166560, 9)
Date range    : 2005-01-01 00:00:00  →  2023-12-31 23:00:00
Missing values: 0

             datetime  Temp_C  Dew_Point_C  RH_percent  Pressure_kPa  \
0 2005-01-01 00:00:00    10.0          0.0         0.0         100.0   
1 2005-01-01 01:00:00    10.0          0.0         0.0         100.0   
2 2005-01-01 02:00:00    10.0          0.0         0.0         100.0   

   Wind_Dir_deg  Wind_Speed_mps  Visibility_km  Precip_cm  
0          98.6             0.0            4.2        0.0  
1          98.6            20.0            5.2        0.0  
2          98.6             0.0            4.7        0.0  


In [4]:
# Linear interpolation for small gaps; forward/back-fill residuals
# (mirrors the cleaning step in the Indiana notebook)
df = df.interpolate(method='linear', limit=3).ffill().bfill()

assert df.isnull().sum().sum() == 0, "NaNs remain after cleaning — inspect the data!"
print("No missing values after cleaning.")


No missing values after cleaning.


## 4. Feature Engineering

All transformations are applied in the **exact same order** as in the training notebook.

### 4a. Cyclical Time Encodings

In [5]:
dt = df['datetime']

df['hour_sin']      = np.sin(2 * np.pi * dt.dt.hour       / 24)
df['hour_cos']      = np.cos(2 * np.pi * dt.dt.hour       / 24)
df['dayofweek_sin'] = np.sin(2 * np.pi * dt.dt.dayofweek  / 7)
df['dayofweek_cos'] = np.cos(2 * np.pi * dt.dt.dayofweek  / 7)
df['month_sin']     = np.sin(2 * np.pi * dt.dt.month      / 12)
df['month_cos']     = np.cos(2 * np.pi * dt.dt.month      / 12)
df['dayofyear_sin'] = np.sin(2 * np.pi * dt.dt.dayofyear  / 365)
df['dayofyear_cos'] = np.cos(2 * np.pi * dt.dt.dayofyear  / 365)

print("Cyclical features created:")
print(df[['hour_sin','hour_cos','dayofweek_sin','dayofweek_cos',
          'month_sin','month_cos','dayofyear_sin','dayofyear_cos']].head(3))


Cyclical features created:
   hour_sin  hour_cos  dayofweek_sin  dayofweek_cos  month_sin  month_cos  \
0  0.000000  1.000000      -0.974928      -0.222521        0.5   0.866025   
1  0.258819  0.965926      -0.974928      -0.222521        0.5   0.866025   
2  0.500000  0.866025      -0.974928      -0.222521        0.5   0.866025   

   dayofyear_sin  dayofyear_cos  
0       0.017213       0.999852  
1       0.017213       0.999852  
2       0.017213       0.999852  


### 4b. Binary Calendar Flags

In [6]:
df['is_weekend']      = (dt.dt.dayofweek >= 5).astype(int)
df['is_business_hour'] = (
    (dt.dt.hour >= 8) & (dt.dt.hour <= 18) & (dt.dt.dayofweek < 5)
).astype(int)

print("is_weekend sample      :", df['is_weekend'].value_counts().to_dict())
print("is_business_hour sample:", df['is_business_hour'].value_counts().to_dict())


is_weekend sample      : {0: 118944, 1: 47616}
is_business_hour sample: {0: 112044, 1: 54516}


### 4c. Temperature × Humidity Interaction

In [7]:
df['temp_rh_interaction'] = df['Temp_C'] * df['RH_percent']

print("temp_rh_interaction sample:")
print(df['temp_rh_interaction'].describe())


temp_rh_interaction sample:
count    166560.000000
mean         38.498590
std          69.922263
min        -256.000000
25%           0.000000
50%          12.600000
75%          83.000000
max         272.000000
Name: temp_rh_interaction, dtype: float64


### 4d. Lag & Rolling Features — ⚠️ Important Note

In the training notebook, `lag_*` and `roll_*` features were derived from **`aep_mw`** (energy consumption). The Kentucky dataset has **no energy consumption column**, so we cannot compute these features from real data.

**Two options:**

| Option | What it does | When to use |
|--------|-------------|-------------|
| **A — Zero placeholders** *(default below)* | Fills all 10 lag/roll columns with 0 | Quick check of weather-only prediction quality; shows model degradation without energy history |
| **B — If you have Kentucky energy data** | Merge it in and compute lags/rolling exactly as in training | Required for a fair apples-to-apples comparison |

The cell below implements **Option A** by default. To use Option B, provide a Kentucky energy CSV with a `datetime` and `aep_mw` column and un-comment the Option B block.

In [8]:
# ── OPTION A: Zero placeholders (no Kentucky energy data available) ─────────
lag_cols  = ['lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'lag_48h', 'lag_168h']
roll_cols = ['roll_mean_24h', 'roll_std_24h', 'roll_mean_168h', 'roll_std_168h']

for col in lag_cols + roll_cols:
    df[col] = 0.0

print("Lag/rolling columns set to zero (Option A).")
print("⚠️  This will affect prediction accuracy — see notebook notes above.")

# ── OPTION B: Compute from real Kentucky energy data ─────────────────────────
# Un-comment and fill in path if you have Kentucky energy consumption data.
#
# KENTUCKY_ENERGY_PATH = 'kentucky_energy.csv'   # must have 'datetime' & 'aep_mw' columns
# energy_ky = pd.read_csv(KENTUCKY_ENERGY_PATH, parse_dates=['datetime'])
# energy_ky['datetime'] = energy_ky['datetime'].dt.round('h')
# df = pd.merge(df, energy_ky[['datetime','aep_mw']], on='datetime', how='inner')
# df = df.sort_values('datetime').reset_index(drop=True)
#
# for lag in [1, 2, 3, 24, 48, 168]:
#     df[f'lag_{lag}h'] = df['aep_mw'].shift(lag)
#
# for window in [24, 168]:
#     shifted = df['aep_mw'].shift(1)
#     df[f'roll_mean_{window}h'] = shifted.rolling(window).mean()
#     df[f'roll_std_{window}h']  = shifted.rolling(window).std()
#
# df.dropna(inplace=True)
# df.reset_index(drop=True, inplace=True)
# print(f"Option B: lag/rolling features computed. Shape after dropna: {df.shape}")


Lag/rolling columns set to zero (Option A).
⚠️  This will affect prediction accuracy — see notebook notes above.


## 5. Validate Feature Matrix

In [9]:
# Verify all 29 features are present and in the right order
missing_cols = [c for c in FEATURE_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

print("✅ All 29 feature columns present.")
print()
print("Feature column order:")
for i, col in enumerate(FEATURE_COLS):
    print(f"  {i+1:02d}. {col}")


✅ All 29 feature columns present.

Feature column order:
  01. Temp_C
  02. Dew_Point_C
  03. RH_percent
  04. Pressure_kPa
  05. Wind_Dir_deg
  06. Wind_Speed_mps
  07. Visibility_km
  08. Precip_cm
  09. hour_sin
  10. hour_cos
  11. dayofweek_sin
  12. dayofweek_cos
  13. month_sin
  14. month_cos
  15. dayofyear_sin
  16. dayofyear_cos
  17. is_weekend
  18. is_business_hour
  19. lag_1h
  20. lag_2h
  21. lag_3h
  22. lag_24h
  23. lag_48h
  24. lag_168h
  25. roll_mean_24h
  26. roll_std_24h
  27. roll_mean_168h
  28. roll_std_168h
  29. temp_rh_interaction


In [10]:
# Quick sanity checks on value ranges
print("=== Feature sanity check ===")
print(f"Cyclical features in [-1, 1]: {df[['hour_sin','hour_cos','month_sin']].abs().max().max():.4f} (should be ≤1.0)")
print(f"is_weekend unique values     : {sorted(df['is_weekend'].unique())}")
print(f"is_business_hour unique vals : {sorted(df['is_business_hour'].unique())}")
print(f"temp_rh_interaction range    : [{df['temp_rh_interaction'].min():.2f}, {df['temp_rh_interaction'].max():.2f}]")
print(f"\nFinal dataframe shape: {df.shape}")
print(f"Date range            : {df['datetime'].min()}  →  {df['datetime'].max()}")


=== Feature sanity check ===
Cyclical features in [-1, 1]: 1.0000 (should be ≤1.0)
is_weekend unique values     : [np.int64(0), np.int64(1)]
is_business_hour unique vals : [np.int64(0), np.int64(1)]
temp_rh_interaction range    : [-256.00, 272.00]

Final dataframe shape: (166560, 30)
Date range            : 2005-01-01 00:00:00  →  2023-12-31 23:00:00


## 6. Scale Features Using the Saved Indiana Scaler

> The feature scaler was **fit on Indiana training data only** — we simply `transform` here (no fitting), which is the correct way to evaluate on a new dataset.

In [11]:
feature_scaler = joblib.load(FEATURE_SCALER_PATH)
target_scaler  = joblib.load(TARGET_SCALER_PATH)

X_raw = df[FEATURE_COLS].values
X_scaled = feature_scaler.transform(X_raw)

print(f"X_scaled shape : {X_scaled.shape}")
print(f"X_scaled mean  : {X_scaled.mean():.4f}  (should be near 0 if distributions match)")
print(f"X_scaled std   : {X_scaled.std():.4f}   (should be near 1 if distributions match)")


ValueError: X has 29 features, but RobustScaler is expecting 45 features as input.

## 7. Create LSTM Sequences

Same `create_sequences` function as training. Look-back = 168 hours (1 week).

In [ ]:
def create_sequences(X, seq_len: int):
    """Slide a window of seq_len steps; no target (y) — we're doing inference."""
    xs = []
    for i in range(seq_len, len(X)):
        xs.append(X[i - seq_len : i])
    return np.array(xs)


X_seq = create_sequences(X_scaled, SEQ_LEN)

print(f"Sequence array shape : {X_seq.shape}")
print(f"  → (n_samples={X_seq.shape[0]}, seq_len={X_seq.shape[1]}, n_features={X_seq.shape[2]})")
print()
# Dates aligned to sequences (first SEQ_LEN rows consumed as look-back)
seq_dates = df['datetime'].values[SEQ_LEN:]
print(f"Prediction timestamps: {seq_dates[0]}  →  {seq_dates[-1]}")


## 8. Save Engineered Feature Matrix

In [ ]:
# Save the flat (pre-sequencing) feature matrix as CSV for inspection / re-use
df[['datetime'] + FEATURE_COLS].to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Engineered feature matrix saved → {OUTPUT_CSV_PATH}")
print(f"Shape: {df[['datetime'] + FEATURE_COLS].shape}")

# Also save numpy arrays for direct model loading
np.save('../data/processed/X_kentucky_seq.npy',   X_seq)
np.save('../data/processed/seq_dates_kentucky.npy', seq_dates.astype(str))
print("\nNumPy arrays saved:")
print("  X_kentucky_seq.npy    — shape", X_seq.shape)
print("  seq_dates_kentucky.npy")


## 9. Run Model Evaluation (if model file available)

In [ ]:
# ── Un-comment this block if you have the trained model available ────────────
# import tensorflow as tf
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
#
# MODEL_PATH = 'indiana_lstm_final.keras'
# KENTUCKY_ENERGY_PATH = 'kentucky_energy.csv'   # needed for true y values
#
# model = tf.keras.models.load_model(MODEL_PATH)
#
# y_pred_scaled = model.predict(X_seq, batch_size=512, verbose=1)
# y_pred = target_scaler.inverse_transform(y_pred_scaled).flatten()
#
# # Load true Kentucky energy values (aligned to sequence dates)
# energy_ky = pd.read_csv(KENTUCKY_ENERGY_PATH, parse_dates=['datetime'])
# energy_ky['datetime'] = energy_ky['datetime'].dt.round('h')
# energy_ky = energy_ky.set_index('datetime')
# y_true = energy_ky.loc[seq_dates, 'aep_mw'].values
#
# mae  = mean_absolute_error(y_true, y_pred)
# rmse = np.sqrt(mean_squared_error(y_true, y_pred))
# mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
# r2   = r2_score(y_true, y_pred)
#
# print('═' * 45)
# print('  KENTUCKY EVALUATION METRICS')
# print('═' * 45)
# print(f'  MAE  : {mae:>10,.2f} MW')
# print(f'  RMSE : {rmse:>10,.2f} MW')
# print(f'  MAPE : {mape:>10.2f} %')
# print(f'  R²   : {r2:>10.4f}')
# print('═' * 45)

print("Un-comment the block above once you have the model + Kentucky energy data.")


## Summary

| Step | Status |
|------|--------|
| Load & clean Kentucky weather | ✅ Done |
| Cyclical time features (8 cols) | ✅ Done |
| Binary calendar flags (2 cols) | ✅ Done |
| Temp × RH interaction (1 col) | ✅ Done |
| Lag features — `lag_1h` … `lag_168h` (6 cols) | ⚠️ Zero-filled (no energy data) |
| Rolling features — mean/std 24h & 168h (4 cols) | ⚠️ Zero-filled (no energy data) |
| Feature scaler applied | ✅ Done (Indiana scaler, transform only) |
| LSTM sequences created (SEQ_LEN=168) | ✅ Done |
| Saved: CSV + `.npy` arrays | ✅ Done |

**Next step**: Provide Kentucky energy consumption data (with `datetime` and `aep_mw` columns) and follow Option B in Section 4d for a complete, fair evaluation.